# Trees and ensembles: learning rules for resale prices

## Introduction

A laptop can lose value in jumps rather than along a straight line. How can a model learn those jumps? This notebook revisits *Tree-Based Algorithms* through worked examples of regression-tree rules and a small boosted model.

The examples show how a tree chooses a split, forms a leaf prediction and limits its size. They also reconstruct boosting updates and assess practical resale-price models on reserved cases. Coding exercises at the end build on these demonstrations.

**Outline:** [Key Concepts](#key-concepts) · [Tree-based forecasting](#tree-based-forecasting) · [Forecast combination and ensemble learning](#forecast-combination-and-ensemble-learning) · [Summary](#summary) · [Tasks](#tasks)

**Lecture connection:** slides 7–28 (split search and leaf forecasts), 34–40 (complexity and classification trees), and 43–68 (ensembles and gradient boosting).

We'll use **pandas** to work with the resale-price data, **NumPy** to calculate candidate splits, and **Matplotlib** to visualize the results. **Scikit-learn** provides the tree and ensemble models. The examples load datasets from the course's `data/` folder.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor, plot_tree, export_text
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

## Key Concepts

A *regression tree* partitions feature space using successive questions such as “is age at most 19.5 months?” Each terminal node, or *leaf*, predicts the mean target among its training observations.

For a local set of row indices $\mathcal{R}$, let $\overline{y}_{\mathcal{R}}$ denote the mean observed resale price. A candidate split creates nonempty left and right groups, $\mathcal{R}_{\mathrm{left}}$ and $\mathcal{R}_{\mathrm{right}}$. Its sum of squared residuals is

$$\sum_{i\in\mathcal{R}_{\mathrm{left}}}(y_i-\overline{y}_{\mathcal{R}_{\mathrm{left}}})^2
+\sum_{i\in\mathcal{R}_{\mathrm{right}}}(y_i-\overline{y}_{\mathcal{R}_{\mathrm{right}}})^2.$$

Lower values mean more homogeneous prices within the children. This is a *greedy* choice at the current node; it does not guarantee the globally best full tree. With $n$ distinct values of one feature there are $n-1$ midpoint candidates.

An *ensemble* combines several models. Bagging and random forests average predictions from separately grown trees. *Gradient boosting* builds trees sequentially to improve a running prediction. Under squared-error loss, each added tree learns the previous model's residuals. Tree predictions are piecewise constant; a weighted sum of tree predictions is still piecewise constant.

## Tree-based forecasting

### Recreate the lecture's small dataset

![Tree lecture slide 9: successive age questions lead to constant resale-price forecasts in the leaves.](../resources/4_tree_rules.png)

The next cell loads the 13 age/price pairs from slide 7, supplied in a CSV. This tiny example helps you see the algorithm; fitting it is not an assessment of generalization. Prices are in USD and age is in months.

In [ ]:
lecture = pd.read_csv('../data/tree_lecture_resale.csv')
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(lecture['age_months'], lecture['resale_price_usd'])
ax.set(xlabel='Age (months)', ylabel='Resale price (USD)', title='The lecture example')
plt.tight_layout()
plt.show()

### Find the first split

The split search sorts the ages and uses midpoints between neighboring observations. For each candidate, it computes the mean price in each group and then the total squared residuals. The threshold itself has units of months; the objective has units of USD squared.

Scikit-learn routes values **at most** the threshold to the left and larger values to the right. The slide uses `<`; no observed age equals these midpoint thresholds, so both conventions give the same groups here.

In [ ]:
ages = np.sort(lecture['age_months'].unique())
thresholds = (ages[:-1] + ages[1:]) / 2
split_results = []
for threshold in thresholds:
    left = lecture.loc[lecture['age_months'] <= threshold, 'resale_price_usd']
    right = lecture.loc[lecture['age_months'] > threshold, 'resale_price_usd']
    squared_residuals = ((left - left.mean()) ** 2).sum() + ((right - right.mean()) ** 2).sum()
    split_results.append({'threshold_months': threshold, 'squared_residuals': squared_residuals})
split_table = pd.DataFrame(split_results)
split_table.sort_values('squared_residuals').head().round(2)

In [ ]:
stump = DecisionTreeRegressor(max_depth=1, random_state=42)
stump.fit(lecture[['age_months']], lecture['resale_price_usd'])
print(f'Best manual threshold: {split_table.loc[split_table["squared_residuals"].idxmin(), "threshold_months"]:.1f} months')
print(f'Scikit-learn root threshold: {stump.tree_.threshold[0]:.1f} months')
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(split_table['threshold_months'], split_table['squared_residuals'], marker='o')
ax.set(xlabel='Candidate age threshold (months)', ylabel='Sum of squared residuals (USD²)',
       title='Choose the split with the smallest objective')
plt.tight_layout()
plt.show()

Both approaches select 19.5 months. `tree_` exposes the fitted structure; element 0 is the root. `.fit` learns split rules, while `.predict` applies those rules to new rows.

### Read a four-leaf tree

`max_leaf_nodes=4` limits this tree to four terminal groups. `plot_tree` draws the learned rules and `export_text` prints them. Read the tree from the root downward; at each split, the left branch satisfies `<=`.

In [ ]:
small_tree = DecisionTreeRegressor(max_leaf_nodes=4, random_state=42)
small_tree.fit(lecture[['age_months']], lecture['resale_price_usd'])
fig, ax = plt.subplots(figsize=(10, 5))
plot_tree(small_tree, feature_names=['Age (months)'], filled=True,
          impurity=False, rounded=True, precision=1, ax=ax)
plt.tight_layout()
plt.show()
print(export_text(small_tree, feature_names=['age_months']))

For these 13 rows, the four leaf means are 983.33, 833.33, 425.00 and 100.00 USD. These differ from the labels in the lecture schematic; the actual training means determine our fitted forecasts.

`apply` returns the leaf identifier for each row. The next cell uses it to identify the training observations that supply the forecast for a new 25-month-old laptop. Leaf identifiers are internal node numbers, not ordered risk or price categories.

In [ ]:
new_laptop = pd.DataFrame({'age_months': [25.0]})
leaf_id = small_tree.apply(new_laptop)[0]
training_leaf = small_tree.apply(lecture[['age_months']])
leaf_prices = lecture.loc[training_leaf == leaf_id, 'resale_price_usd']
print('Prices in this leaf:', leaf_prices.to_list())
print(f'Leaf mean: {leaf_prices.mean():.2f} USD')
print(f'Model forecast: {small_tree.predict(new_laptop)[0]:.2f} USD')

In [ ]:
age_grid = pd.DataFrame({'age_months': np.linspace(3, 48, 200)})
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(lecture['age_months'], lecture['resale_price_usd'], label='Training observations')
ax.plot(age_grid['age_months'], small_tree.predict(age_grid), color='darkorange', label='Four-leaf forecast')
ax.axvline(lecture['age_months'].max(), linestyle='--', color='gray', label='Oldest observed laptop')
ax.set(xlabel='Age (months)', ylabel='Resale price (USD)', title='Constant forecasts within leaves')
ax.legend()
plt.tight_layout()
plt.show()

Notice the steps and the flat extension past the oldest observation. A tree cannot continue a depreciation slope outside the observed range; a plausible forecast there requires additional evidence or assumptions.

### Control complexity on the course resale data

![Tree lecture slide 36: maximum depth, leaf count and minimum node size constrain tree growth.](../resources/4_tree_pruning.png)

The next example uses the larger course dataset to predict price **at equipment return, before sale**, from retail price, actual lease duration and measured battery capacity. These last two inputs would not be available at contract signing; the prediction time matters.

Training data supply the estimated splits, validation data support a comparison of three depths, and the test data remain reserved for the final assessment. Trees do not need standardization: their rules compare one feature with a threshold in that feature's own units.

In [ ]:
resale = pd.read_csv('../data/resale_price_dataset.csv')
features = ['Retail price', 'Actual Lease Duration (months)', 'Battery capacity (%)']
X = resale[features]
y = resale['Observed resale price']
X_dev, X_test, y_dev, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_valid, y_train, y_valid = train_test_split(X_dev, y_dev, test_size=0.25, random_state=42)
print(f'Train: {len(X_train)}; validation: {len(X_valid)}; test: {len(X_test)}')
X_train.describe().round(1)

In [ ]:
depth_results = []
for depth in [2, 4, 8]:
    candidate = DecisionTreeRegressor(max_depth=depth, min_samples_leaf=20, random_state=42)
    candidate.fit(X_train, y_train)
    depth_results.append({
        'depth': depth,
        'training_mae': mean_absolute_error(y_train, candidate.predict(X_train)),
        'validation_mae': mean_absolute_error(y_valid, candidate.predict(X_valid)),
        'leaves': candidate.get_n_leaves()
    })
depth_table = pd.DataFrame(depth_results).set_index('depth')
depth_table.round(2)

The validation column provides the basis for the depth choice later in the demonstration. Lower training error alone does not justify a larger tree. `min_samples_leaf=20` provides another growth constraint; this is *pre-pruning*, not post-pruning of an already fitted tree. We do not claim that these three settings exhaust the possible choices.

For a classification target, the interface becomes `DecisionTreeClassifier`; leaves report class proportions and splits use an impurity measure such as Gini impurity rather than squared price residuals. This distinguishes a classification tree for the credit data from the regression trees fitted here.

## Forecast combination and ensemble learning

### Separate models versus sequential corrections

A *random forest* grows trees from bootstrap samples, considering a random subset of features at each split, then averages their predictions. The trees are not fitted to one another's errors. Correlated trees limit the benefit of averaging.

### Inspect a random forest forecast

`RandomForestRegressor` uses the familiar `.fit` and `.predict` interface. Our example fits 50 trees, allows two candidate features per split, and keeps at least ten observations in each leaf. Bootstrapping is enabled by default.

For one validation row, the next cell reconstructs the forest forecast as the mean of the constituent tree forecasts. `estimators_` stores the trees. Individual trees were fitted internally on arrays, so the code passes `.to_numpy()` when inspecting them directly. This calculation illustrates aggregation, not variability across training samples.

In [ ]:
forest = RandomForestRegressor(
    n_estimators=50, max_features=2, min_samples_leaf=10, random_state=42
)
forest.fit(X_train, y_train)
validation_laptop = X_valid.iloc[[0]]
tree_forecasts = np.array([
    tree.predict(validation_laptop.to_numpy())[0] for tree in forest.estimators_
])
print(f'Mean of tree forecasts: {tree_forecasts.mean():.2f} USD')
print(f'Forest prediction: {forest.predict(validation_laptop)[0]:.2f} USD')

### Sequential corrections with boosting

*Gradient boosting* instead corrects an existing forecast. For squared-error regression, define $f_m$ as the running model after stage $m$, $h_m$ as the next fitted tree, and $\nu$ as a local learning-rate symbol:

$$f_0(\mathbf{x})=\overline{y}_{\mathrm{train}},\qquad
r_i^{(m)}=y_i-f_{m-1}(\mathbf{x}_i),\qquad
f_m(\mathbf{x})=f_{m-1}(\mathbf{x})+\nu h_m(\mathbf{x}).$$

The tree $h_m$ fits the current residuals. This residual interpretation applies directly to squared error; other losses use different negative gradients.

![Tree lecture slide 67: boosting starts from a mean forecast and adds trees fitted to the remaining errors.](../resources/4_boosting.png)

### Follow the first two updates

The tiny age/price example makes the arithmetic visible. `init_.predict` supplies the initial mean forecast. `estimators_[m, 0]` retrieves the regression tree at stage `m` (Python counts from zero). Each tree's prediction is multiplied by `learning_rate` before being added to the running forecast.

In [ ]:
toy_boost = GradientBoostingRegressor(
    n_estimators=2, max_depth=1, learning_rate=0.5, random_state=42
)
lecture_X = lecture[['age_months']]
toy_boost.fit(lecture_X, lecture['resale_price_usd'])
initial = toy_boost.init_.predict(lecture_X)
first_correction = toy_boost.estimators_[0, 0].predict(lecture_X.to_numpy())
after_first = initial + toy_boost.learning_rate * first_correction
second_correction = toy_boost.estimators_[1, 0].predict(lecture_X.to_numpy())
after_second = after_first + toy_boost.learning_rate * second_correction
pd.DataFrame({
    'age_months': lecture['age_months'],
    'actual_price': lecture['resale_price_usd'],
    'initial_forecast': initial,
    'residual_before_stage_1': lecture['resale_price_usd'] - initial,
    'stage_1_forecast': after_first,
    'stage_2_manual': after_second,
    'stage_2_sklearn': toy_boost.predict(lecture_X)
}).round(2)

The manual two-stage calculation agrees with `.predict`. A correction can be negative: it adjusts a forecast rather than predicting a complete resale price on its own.

### Fit and assess the practical models

The boosting configuration below stays fixed, while validation MAE determines the single-tree depth. The code fits that tree, the boosted model and a mean-price baseline on all development data before evaluating the reserved test set. This comparison provides a final assessment; further hyperparameter choices based on these results would compromise its independence.

In [ ]:
chosen_depth = int(depth_table['validation_mae'].idxmin())
final_models = {
    'Mean-price baseline': DummyRegressor(strategy='mean'),
    'Validation-selected tree': DecisionTreeRegressor(
        max_depth=chosen_depth, min_samples_leaf=20, random_state=42
    ),
    'Gradient boosting': GradientBoostingRegressor(
        n_estimators=100, max_depth=2, learning_rate=0.05, random_state=42
    )
}
final_results = []
for name, model in final_models.items():
    model.fit(X_dev, y_dev)
    final_results.append({'model': name, 'test_mae_usd': mean_absolute_error(y_test, model.predict(X_test))})
final_table = pd.DataFrame(final_results).set_index('model')
final_table.round(2)

MAE is measured in USD per laptop. The baseline puts both learned models' errors into context, although one split's ranking does not establish a universal advantage. A lower price error does not by itself establish higher resale profit.

## Summary

- A regression tree learns feature thresholds and predicts training means within leaves.
- Split search is greedy; depth and minimum leaf size control flexibility.
- Tree forecasts are constant inside leaves, including extensions beyond observed feature ranges.
- Random forests average separately trained trees; boosting adds sequential corrections.
- Validation evidence guides settings; reserved test observations provide the final assessment.

Here's a quick reference for the new Python tools introduced in this notebook:

| Python pattern | What it helps you do |
|---|---|
| `np.sort`, `.unique()` | Construct candidate thresholds from ordered distinct feature values |
| `DecisionTreeRegressor` | Learn regression-tree split rules |
| `plot_tree`, `export_text` | Inspect the fitted tree |
| `model.apply(X)` | Find the leaf assigned to each observation |
| `mean_absolute_error` | Measure price error in USD |
| `RandomForestRegressor`, `GradientBoostingRegressor` | Fit averaging and sequential-correction tree ensembles |
| `.estimators_`, `.init_` | Inspect constituent trees and the boosting initial forecast |
| `.idxmin()` | Select the lowest validation error |

## Tasks

### Task 1: Verify another leaf

Use a new age of 10 months. Find its leaf, list that leaf's training prices and reconstruct its forecast. Explain why two different ages can receive the same prediction.

*Hint:* reuse `apply` and the training-leaf selection.

In [ ]:
# YOUR CODE: reconstruct the forecast for a 10-month-old laptop.

### Task 2: Audit a candidate split

Select the threshold 30 months from `split_table`. Calculate both child means, their observation counts and their total squared residuals. Compare with the stored objective and explain why it loses to the best split.

In [ ]:
# YOUR CODE: audit the 30-month split.

### Task 3: Inspect a boosting correction

Plot `first_correction` against age, alongside the initial residuals. Add a horizontal line at zero. Explain what positive and negative tree outputs do to the running price forecast. Keep the final test data unchanged.

In [ ]:
# YOUR CODE: visualize the first residual correction.